# MAD v2 — Multi-Agent Debate Pipeline
### Three-Agent: Verifier + Adversarial Auditor + Calibrator
**Fixes in v2:** anti-sycophancy prompts · awq_marlin · retry logic · confidence clamping · GRPO export

**Run cells top to bottom. Cell 16 is the main run — it is idempotent (safe to re-run).**

In [ ]:
# CELL 1 — Install dependencies
%%capture
!pip install vllm aiohttp nest_asyncio tenacity pydantic
print('Install complete')

In [ ]:
# CELL 2 — GPU check + fix Colab async
import subprocess, nest_asyncio, logging

nest_asyncio.apply()

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'GPU: {gpu}')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('MAD')
print('\u2713 Logging configured')
print('\u2713 nest_asyncio applied')

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB
✓ Logging configured
✓ nest_asyncio applied


In [ ]:
# CELL 3 — Upload DB file
from google.colab import files
import sqlite3

print('Upload your .db file (the one with queries + claims already populated):')
uploaded = files.upload()

db_files = [f for f in uploaded.keys() if f.endswith('.db')]
if not db_files:
    raise ValueError('No .db file found in upload')

DB_PATH = f'/content/{db_files[0]}'
print(f'\nDB path: {DB_PATH}')

conn = sqlite3.connect(DB_PATH)
q_count  = conn.execute('SELECT COUNT(*) FROM queries').fetchone()[0]
c_count  = conn.execute('SELECT COUNT(*) FROM claims').fetchone()[0]
ao_count = conn.execute('SELECT COUNT(*) FROM agent_outputs').fetchone()[0]
ad_count = conn.execute('SELECT COUNT(*) FROM agent_deltas').fetchone()[0]
conn.close()

print(f'  queries:       {q_count}   (expected 50)')
print(f'  claims:        {c_count}   (expected 414)')
print(f'  agent_outputs: {ao_count}  (0 = fresh start, >0 = resuming)')
print(f'  agent_deltas:  {ad_count}')
print('\u2713 DB loaded')

Upload your .db file (the one with queries + claims already populated):


Saving mad_before_phase1_5090_ragfix_01_.db to mad_before_phase1_5090_ragfix_01_.db

DB path: /content/mad_before_phase1_5090_ragfix_01_.db
  queries:       50   (expected 50)
  claims:        414   (expected 414)
  agent_outputs: 0  (0 = fresh start, >0 = resuming)
  agent_deltas:  0
✓ DB loaded


In [ ]:
# CELL 4 — Config (all tunable constants in one place)

MODEL_NAME  = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
VLLM_PORT   = 8001
VLLM_URL    = f'http://localhost:{VLLM_PORT}/v1'
SERVED_NAME = 'agents'
LOG_PATH    = '/content/vllm_v2.log'

# Context limits
MAX_CHUNK_CHARS = 800   # per chunk text (~200 tokens each)
MAX_CLAIM_CHARS = 300   # claim text cap
MAX_PEER_CHARS  = 600   # peer reasoning shown in Round 1

# v2: per-round temperatures — R1 higher to break consensus
TEMPERATURES_R0 = {'agent_a': 0.3, 'agent_b': 0.8, 'agent_c': 0.5}
TEMPERATURES_R1 = {'agent_a': 0.4, 'agent_b': 0.9, 'agent_c': 0.5}

# Concurrency: N claims x 3 agents = N*3 concurrent calls
CLAIM_CONCURRENCY = 4   # 4 x 3 = 12 concurrent — matches --max-num-seqs 12
MAX_TOKENS_OUT    = 600
VLLM_TIMEOUT      = 150

# v2: retry config
MAX_PARSE_RETRIES = 3
RETRY_BACKOFF     = [2, 5, 10]  # seconds between retries

# v2: confidence clamping — prevents 0.0 and 1.0 from destroying Brier signal
CONF_MAX = 0.95
CONF_MIN = 0.05

AGENT_ROLES   = ['agent_a', 'agent_b', 'agent_c']
DEBATER_LABEL = {'agent_a': 'Debater 1', 'agent_b': 'Debater 2', 'agent_c': 'Debater 3'}

print('\u2713 Config set')
print(f'  Model:       {MODEL_NAME}')
print(f'  Concurrency: {CLAIM_CONCURRENCY} claims x 3 agents = {CLAIM_CONCURRENCY*3} concurrent calls')
print(f'  Retries:     up to {MAX_PARSE_RETRIES} attempts per call')

✓ Config set
  Model:       Qwen/Qwen2.5-14B-Instruct-AWQ
  Concurrency: 4 claims x 3 agents = 12 concurrent calls
  Retries:     up to 3 attempts per call


In [ ]:
# CELL 5 — Start vLLM server
# v2 changes vs v1:
#   --quantization awq_marlin   (faster inference kernel vs plain awq)
#   --generation-config vllm    (disables model baked-in temp=0.7 override)
import subprocess, time

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model',                  MODEL_NAME,
    '--quantization',           'awq_marlin',  # v2: faster than awq
    '--max-model-len',          '6144',
    '--gpu-memory-utilization', '0.88',
    '--max-num-seqs',           '12',
    '--enable-prefix-caching',
    '--port',                   str(VLLM_PORT),
    '--served-model-name',      SERVED_NAME,
    '--trust-remote-code',
    '--dtype',                  'float16',
    '--generation-config',      'vllm',  # v2: our temperatures now actually apply
]

print('Launching vLLM server...')
print('Command:', ' '.join(cmd))

server_log  = open(LOG_PATH, 'w')
server_proc = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT)
print(f'Server PID: {server_proc.pid}')
print(f'Log: {LOG_PATH}')
print('\u2192 Run CELL 6 to wait for ready (~3-4 min)')

Launching vLLM server...
Command: python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-14B-Instruct-AWQ --quantization awq_marlin --max-model-len 6144 --gpu-memory-utilization 0.88 --max-num-seqs 12 --enable-prefix-caching --port 8001 --served-model-name agents --trust-remote-code --dtype float16 --generation-config vllm
Server PID: 2549
Log: /content/vllm_v2.log
→ Run CELL 6 to wait for ready (~3-4 min)


In [ ]:
# CELL 6 — Wait for vLLM server ready
import requests

def wait_for_server(url, timeout_secs=360):
    print('Waiting for vLLM server', end='', flush=True)
    for i in range(timeout_secs // 5):
        try:
            r = requests.get(f'{url}/models', timeout=3)
            if r.status_code == 200:
                models = r.json().get('data', [])
                print(f'\n\u2713 Ready in {i*5}s! Models: {[m["id"] for m in models]}')
                return True
        except Exception:
            pass
        print('.', end='', flush=True)
        time.sleep(5)
    print('\n\u2717 Server failed to start. Check logs:')
    import subprocess
    print(subprocess.run(['tail', '-40', LOG_PATH], capture_output=True, text=True).stdout)
    return False

ready = wait_for_server(VLLM_URL)

if ready:
    # Show key server stats from log
    import subprocess
    kv_line = subprocess.run(
        ['grep', '-i', 'kv cache', LOG_PATH],
        capture_output=True, text=True
    ).stdout.strip()
    if kv_line:
        print(f'KV cache info: {kv_line[-200:]}')

Waiting for vLLM server..................................
✓ Ready in 170s! Models: ['agents']
KV cache info: _cache_utils.py:1708] GPU KV cache size: 129,184 tokens
(EngineCore pid=2863) INFO 05-06 07:55:36 [core.py:299] init engine (profile, create kv cache, warmup model) took 56.31 s (compilation: 38.93 s)


In [ ]:
# CELL 7 — Pydantic schemas + bias control
from pydantic import BaseModel, Field
from typing import List
from enum import Enum

class AgentVerdict(str, Enum):
    SUPPORTED     = 'SUPPORTED'
    PARTIAL       = 'PARTIAL'
    NOT_SUPPORTED = 'NOT_SUPPORTED'
    IDK           = 'IDK'

class AgentRole(str, Enum):
    AGENT_A = 'agent_a'
    AGENT_B = 'agent_b'
    AGENT_C = 'agent_c'

class EvidenceCitation(BaseModel):
    chunk_id:       str
    relevant_quote: str = ''  # v2 fix: was crashing when model omits this field

class AgentOutputFull(BaseModel):
    """Full output — written to SQLite. NEVER passed to peers or judge."""
    agent_role:          AgentRole
    round_num:           int
    verdict:             AgentVerdict
    reasoning:           str
    evidence_cited:      List[EvidenceCitation]
    confidence_internal: float = Field(ge=0.0, le=1.0)

class AgentOutputStripped(BaseModel):
    """Stripped output — safe for inter-agent passing. No confidence, no role."""
    debater_label:  str
    round_num:      int
    verdict:        AgentVerdict
    reasoning:      str
    evidence_cited: List[EvidenceCitation]

def strip_for_peer(output: AgentOutputFull, debater_label: str) -> AgentOutputStripped:
    """THE only function that creates AgentOutputStripped.
    Removes confidence_internal and anonymizes agent identity.
    Bias control choke point — never bypass."""
    return AgentOutputStripped(
        debater_label  = debater_label,
        round_num      = output.round_num,
        verdict        = output.verdict,
        reasoning      = output.reasoning[:MAX_PEER_CHARS],
        evidence_cited = output.evidence_cited,
    )

# Verify bias control
def _test_strip():
    full = AgentOutputFull(
        agent_role=AgentRole.AGENT_A, round_num=0,
        verdict=AgentVerdict.SUPPORTED, reasoning='test',
        evidence_cited=[], confidence_internal=0.85
    )
    stripped = strip_for_peer(full, 'Debater 1')
    s = stripped.model_dump_json()
    assert 'confidence' not in s.lower(), 'LEAK: confidence in stripped output!'
    assert 'agent_a'   not in s.lower(), 'LEAK: agent role in stripped output!'
    print('\u2713 Bias control test passed')

_test_strip()

✓ Bias control test passed


In [ ]:
# CELL 8 — System prompts v2
# v2 changes:
#   - Explicitly penalize PARTIAL as hedge (require specific justification)
#   - confidence 1.0 and 0.0 forbidden in prompt
#   - Agent B pushed harder toward NOT_SUPPORTED when evidence is silent

AGENT_A_SYSTEM = """You are a strict regulatory compliance verifier in a multi-agent debate.

YOUR ROLE: Find the strongest case FOR the claim being correct, based ONLY on the retrieved evidence.

RULES:
1. Analyze the evidence. Find what specifically SUPPORTS the claim.
2. Verdict:
   - SUPPORTED: evidence clearly and directly backs the claim. USE THIS when evidence supports.
   - PARTIAL: ONLY when evidence supports SOME but not ALL parts. You MUST specify exactly what is supported and what is not.
   - NOT_SUPPORTED: evidence contradicts or is completely silent on the claim.
   - IDK: evidence is genuinely too ambiguous. Use rarely.
3. CRITICAL: Do NOT use PARTIAL as a safe default. If the evidence supports the claim, say SUPPORTED.
   Every PARTIAL verdict requires you to name the specific supported part AND the specific missing part.
4. Cite specific chunk_ids. relevant_quote must be actual text copied from the chunk.
5. confidence_internal: your TRUE certainty. NEVER assign 1.0 (always some uncertainty). NEVER assign 0.0.
   Valid range: 0.05 to 0.95.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON object.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Specific analysis citing chunk content...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "exact text from chunk"}],
    "confidence_internal": 0.05-0.95
}"""

AGENT_B_SYSTEM = """You are an adversarial auditor in a multi-agent regulatory compliance debate.

YOUR ROLE: Find what is WRONG, UNSUPPORTED, or MISLEADING about the claim using ONLY the retrieved evidence.

RULES:
1. Analyze the evidence. Find what specifically UNDERMINES the claim.
2. Verdict:
   - NOT_SUPPORTED: evidence contradicts the claim OR is completely silent on it. USE THIS when claim is unsupported.
   - PARTIAL: ONLY when part of the claim is wrong or missing. Name the specific wrong part AND the valid part.
   - SUPPORTED: if the claim is genuinely well-supported despite your scrutiny. Be honest.
   - IDK: only when evidence is truly ambiguous.
3. CRITICAL: Do NOT use PARTIAL as a safe default. If the evidence does not back the claim, say NOT_SUPPORTED.
   Silence in the evidence = NOT_SUPPORTED, not PARTIAL.
4. Look for: scope errors, unsupported specifics, missing exceptions, overgeneralizations.
5. Cite chunk_ids that FAIL to support or actively contradict the claim.
6. confidence_internal: NEVER 1.0 or 0.0. Valid range: 0.05 to 0.95.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON object.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Specific challenge citing chunk content...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "exact text from chunk"}],
    "confidence_internal": 0.05-0.95
}"""

AGENT_C_SYSTEM = """You are a neutral evidence calibrator in a multi-agent regulatory compliance debate.

YOUR ROLE: Weigh the evidence honestly on BOTH sides. You have NO prior stance.

RULES:
1. Separately analyze: (a) what the evidence supports about the claim, (b) what it undermines.
2. Verdict:
   - SUPPORTED: evidence clearly favors the claim. Say so. Do not hedge.
   - NOT_SUPPORTED: evidence clearly goes against the claim. Say so. Do not hedge.
   - PARTIAL: ONLY when evidence genuinely cuts both ways. Name the supported part AND the unsupported part.
   - IDK: when evidence is truly insufficient to decide.
3. CRITICAL: Do NOT use PARTIAL or IDK as safe defaults. Intellectual honesty means calling clear cases clearly.
   If evidence points clearly one way, commit to SUPPORTED or NOT_SUPPORTED.
4. Cite chunk_ids for BOTH directions when evidence is mixed.
5. confidence_internal: calibrated 0.05-0.95. 0.5 = genuinely 50/50. Not a default hedge.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON object.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Balanced analysis with evidence for and against...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "exact text from chunk"}],
    "confidence_internal": 0.05-0.95
}"""

SYSTEM_PROMPTS = {
    'agent_a': AGENT_A_SYSTEM,
    'agent_b': AGENT_B_SYSTEM,
    'agent_c': AGENT_C_SYSTEM,
}

print('\u2713 System prompts v2 defined')
for role, prompt in SYSTEM_PROMPTS.items():
    print(f'  {role}: {len(prompt)} chars (~{len(prompt)//4} tokens)')

✓ System prompts v2 defined
  agent_a: 1383 chars (~345 tokens)
  agent_b: 1379 chars (~344 tokens)
  agent_c: 1283 chars (~320 tokens)


In [ ]:
# CELL 9 — DB helpers v2
# v2 additions: schema patch for parse_failed/call_failed/retry_count columns
import sqlite3, json, threading, uuid
from datetime import datetime

db_lock = threading.Lock()

# ── Schema patch (adds v2 columns if not present) ────────────────────────────
def apply_schema_patch():
    conn = sqlite3.connect(DB_PATH)
    for col, typ in [('parse_failed', 'INTEGER DEFAULT 0'),
                     ('call_failed',  'INTEGER DEFAULT 0'),
                     ('retry_count',  'INTEGER DEFAULT 0')]:
        try:
            conn.execute(f'ALTER TABLE agent_outputs ADD COLUMN {col} {typ}')
        except sqlite3.OperationalError as e:
            if 'duplicate column' not in str(e).lower():
                logger.warning(f'Schema patch {col}: {e}')
    conn.commit()
    conn.close()
    print('\u2713 Schema patch applied (parse_failed, call_failed, retry_count)')

apply_schema_patch()

# ── Readers ───────────────────────────────────────────────────────────────────
def load_all_queries():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute('SELECT * FROM queries ORDER BY query_id').fetchall()
    conn.close()
    result = []
    for r in rows:
        d = dict(r)
        d['rag_chunks']    = json.loads(d['rag_chunks'])
        d['rag_chunk_ids'] = json.loads(d['rag_chunk_ids'])
        result.append(d)
    return result

def load_claims_for_query(query_id):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        'SELECT * FROM claims WHERE query_id=? ORDER BY claim_index', (query_id,)
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def get_completed_claim_ids():
    """Return set of claim_ids that have R1 outputs for all 3 agents."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("""
        SELECT claim_id FROM agent_outputs
        WHERE round_num = 1
        GROUP BY claim_id
        HAVING COUNT(DISTINCT agent_role) = 3
    """).fetchall()
    conn.close()
    return {r[0] for r in rows}

# ── Writers ───────────────────────────────────────────────────────────────────
def write_agent_output(
    claim_id, agent_role, round_num,
    verdict, reasoning, evidence_cited,
    confidence_internal, raw_response,
    latency_ms, tokens_in, tokens_out,
    parse_failed=False, call_failed=False, retry_count=0
):
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO agent_outputs
            (output_id, claim_id, agent_role, round_num, verdict, reasoning,
             evidence_cited, confidence_internal, raw_response,
             latency_ms, tokens_in, tokens_out, timestamp,
             parse_failed, call_failed, retry_count)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,CURRENT_TIMESTAMP,?,?,?)
        """, (
            str(uuid.uuid4()), claim_id, agent_role, round_num,
            verdict, reasoning,
            json.dumps([e.model_dump() if hasattr(e, 'model_dump') else e
                        for e in evidence_cited]),
            confidence_internal, raw_response[:4000],
            latency_ms, tokens_in, tokens_out,
            int(parse_failed), int(call_failed), retry_count
        ))
        conn.commit()
        conn.close()

def write_agent_delta(
    claim_id, agent_role,
    conf_r0, conf_r1, verdict_r0, verdict_r1
):
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO agent_deltas
            (delta_id, claim_id, agent_role,
             confidence_r0, confidence_r1, delta,
             verdict_r0, verdict_r1, verdict_changed)
            VALUES (?,?,?,?,?,?,?,?,?)
        """, (
            str(uuid.uuid4()), claim_id, agent_role,
            conf_r0, conf_r1, round(conf_r1 - conf_r0, 4),
            verdict_r0, verdict_r1, int(verdict_r0 != verdict_r1)
        ))
        conn.commit()
        conn.close()

# ── Live run stats ────────────────────────────────────────────────────────────
class RunStats:
    def __init__(self):
        self.total   = 0
        self.parse_f = 0
        self.call_f  = 0
        self.verdicts = {'SUPPORTED': 0, 'PARTIAL': 0, 'NOT_SUPPORTED': 0, 'IDK': 0}
        self.retries  = 0

    def record(self, parsed, retry_count=0):
        self.total += 1
        self.retries += retry_count
        if parsed.get('_parse_failed'): self.parse_f += 1
        if parsed.get('_call_failed'):  self.call_f  += 1
        v = parsed.get('verdict', 'IDK')
        self.verdicts[v] = self.verdicts.get(v, 0) + 1

    def summary(self):
        t = self.total or 1
        partial_pct = self.verdicts.get('PARTIAL', 0) / t * 100
        print(f'\n=== Run Stats (so far) ===')
        print(f'  Calls: {self.total}  |  Parse fails: {self.parse_f} ({self.parse_f/t*100:.1f}%)  |  Call fails: {self.call_f}')
        print(f'  Retries used: {self.retries}')
        print(f'  Verdicts: {self.verdicts}')
        print(f'  PARTIAL rate: {partial_pct:.1f}% (v1 was 69% — target <50%)')

run_stats = RunStats()
print('\u2713 DB helpers v2 ready')
print(f'  Queries loaded: {len(load_all_queries())}')

✓ Schema patch applied (parse_failed, call_failed, retry_count)
✓ DB helpers v2 ready
  Queries loaded: 50


In [ ]:
# CELL 10 — vLLM async client v2
# v2: retry on parse failure, backoff on call error, confidence clamping
import aiohttp, asyncio, time, re, json

# ── JSON parser (4 fallback strategies) ──────────────────────────────────────
def parse_agent_json(raw: str) -> dict:
    raw = raw.strip()
    for fn in [
        lambda r: json.loads(r),
        lambda r: json.loads(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', r, re.DOTALL).group(1)),
        lambda r: json.loads(re.search(r'\{.*\}', r, re.DOTALL).group(0)),
    ]:
        try:
            result = fn(raw)
            if isinstance(result, dict) and 'verdict' in result:
                return result
        except Exception:
            continue
    # Regex field extraction as last resort
    vm = re.search(r'"verdict"\s*:\s*"([^"]+)"', raw)
    cm = re.search(r'"confidence_internal"\s*:\s*([0-9.]+)', raw)
    rm = re.search(r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*?)"', raw)
    return {
        'verdict':             vm.group(1) if vm else 'IDK',
        'reasoning':           rm.group(1) if rm else f'[PARSE_FAILED] {raw[:400]}',
        'evidence_cited':      [],
        'confidence_internal': float(cm.group(1)) if cm else 0.5,
        '_parse_failed':       True,
    }

def normalize_verdict(v: str) -> str:
    return {
        'SUPPORTED': 'SUPPORTED', 'SUPPORT': 'SUPPORTED',
        'NOT_SUPPORTED': 'NOT_SUPPORTED', 'NOT SUPPORTED': 'NOT_SUPPORTED',
        'NOTSUPPORTED': 'NOT_SUPPORTED', 'UNSUPPORTED': 'NOT_SUPPORTED',
        'PARTIAL': 'PARTIAL', 'PARTIALLY SUPPORTED': 'PARTIAL', 'PARTIAL SUPPORT': 'PARTIAL',
        'IDK': 'IDK', 'INSUFFICIENT': 'IDK', 'UNKNOWN': 'IDK', 'UNCLEAR': 'IDK',
    }.get(v.upper().strip().replace('-', '_'), 'IDK')

def clamp_conf(c) -> float:
    try:
        return max(CONF_MIN, min(CONF_MAX, float(c)))
    except Exception:
        return 0.5

def safe_evidence(raw_list: list) -> list:
    result = []
    for e in (raw_list or []):
        if not isinstance(e, dict) or 'chunk_id' not in e:
            continue
        result.append(EvidenceCitation(
            chunk_id=str(e.get('chunk_id', 'unknown'))[:100],
            relevant_quote=str(e.get('relevant_quote', e.get('quote', '')))[:300],
        ))
    return result

# ── Async call with retries ───────────────────────────────────────────────────
async def call_agent_with_retry(
    session, system_prompt, user_prompt, temperature, role, claim_id, round_num
):
    """
    Returns: (parsed_dict, raw_text, latency_ms, tokens_in, tokens_out, retry_count)
    Never raises. Returns IDK fallback on total failure.
    """
    last_err_type = None

    for attempt in range(MAX_PARSE_RETRIES):
        # On parse retry: append JSON reminder
        prompt = user_prompt
        if attempt > 0 and last_err_type == 'parse':
            prompt += '\n\nREMINDER: Respond with ONLY a valid JSON object. Start with { end with }. No other text.'

        payload = {
            'model':       SERVED_NAME,
            'messages':    [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': prompt},
            ],
            'temperature': temperature,
            'max_tokens':  MAX_TOKENS_OUT,
            'stream':      False,
        }

        start = time.time()
        try:
            async with session.post(
                f'{VLLM_URL}/chat/completions',
                json=payload,
                timeout=aiohttp.ClientTimeout(total=VLLM_TIMEOUT),
            ) as resp:
                resp.raise_for_status()
                data       = await resp.json()
                latency_ms = int((time.time() - start) * 1000)
                raw_text   = data['choices'][0]['message']['content']
                tokens_in  = data['usage']['prompt_tokens']
                tokens_out = data['usage']['completion_tokens']

        except Exception as e:
            latency_ms = int((time.time() - start) * 1000)
            logger.warning(f'[{role}] R{round_num} call error attempt {attempt+1}: {type(e).__name__}: {e}')
            if attempt < MAX_PARSE_RETRIES - 1:
                await asyncio.sleep(RETRY_BACKOFF[attempt])
                last_err_type = 'call'
                continue
            raw_text = f'[CALL_ERROR] {str(e)}'
            return ({'verdict': 'IDK', 'reasoning': raw_text,
                     'evidence_cited': [], 'confidence_internal': 0.5, '_call_failed': True},
                    raw_text, latency_ms, 0, 0, attempt)

        parsed = parse_agent_json(raw_text)

        if parsed.get('_parse_failed'):
            logger.warning(f'[{role}] R{round_num} parse fail attempt {attempt+1} | {claim_id[:8]} | {raw_text[:60]}')
            if attempt < MAX_PARSE_RETRIES - 1:
                await asyncio.sleep(RETRY_BACKOFF[attempt])
                last_err_type = 'parse'
                continue
            logger.error(f'[{role}] R{round_num} all {MAX_PARSE_RETRIES} retries exhausted | {claim_id[:8]}')
        elif attempt > 0:
            logger.info(f'[{role}] R{round_num} recovered on attempt {attempt+1}')

        # Normalize + clamp
        parsed['verdict']             = normalize_verdict(parsed.get('verdict', 'IDK'))
        parsed['confidence_internal'] = clamp_conf(parsed.get('confidence_internal', 0.5))
        parsed['evidence_cited']      = safe_evidence(parsed.get('evidence_cited', []))

        return parsed, raw_text, latency_ms, tokens_in, tokens_out, attempt

    # Should never reach here
    return ({'verdict': 'IDK', 'reasoning': '[EXHAUSTED]',
              'evidence_cited': [], 'confidence_internal': 0.5},
            '[EXHAUSTED]', 0, 0, 0, MAX_PARSE_RETRIES - 1)

print('\u2713 vLLM client v2 ready (retries, clamping, 4-strategy parser)')

✓ vLLM client v2 ready (retries, clamping, 4-strategy parser)


In [ ]:
# CELL 11 — Prompt builders

def format_chunks(chunks: list, max_chars: int = MAX_CHUNK_CHARS) -> str:
    parts = []
    for i, c in enumerate(chunks):
        text = c['text'][:max_chars]
        if len(c['text']) > max_chars:
            text += '... [truncated]'
        parts.append(
            f"[Chunk {i+1} | ID: {c['chunk_id']} | Source: {c['source_file']}]\n{text}"
        )
    return '\n\n'.join(parts)

def build_round0_prompt(query: str, claim_text: str, chunks: list) -> str:
    return (
        f"USER QUERY: {query}\n\n"
        f"CLAIM TO VERIFY:\n{claim_text[:MAX_CLAIM_CHARS]}\n\n"
        f"RETRIEVED EVIDENCE:\n{format_chunks(chunks)}\n\n"
        f"Provide your independent verdict on this claim."
    )

def build_round1_prompt(
    query: str, claim_text: str, chunks: list,
    peer1: AgentOutputStripped, peer2: AgentOutputStripped
) -> str:
    """v2: anti-sycophancy Round 1 prompt.
    Detects when both peers agree and injects a consensus warning."""

    def fmt_peer(p: AgentOutputStripped) -> str:
        ev = ', '.join(e.chunk_id for e in p.evidence_cited[:3]) or 'none cited'
        return (
            f"Verdict: {p.verdict.value}\n"
            f"Reasoning: {p.reasoning[:MAX_PEER_CHARS]}\n"
            f"Evidence cited: {ev}"
        )

    # Detect R0 consensus → inject extra independence warning
    p1v = peer1.verdict.value if hasattr(peer1.verdict, 'value') else str(peer1.verdict)
    p2v = peer2.verdict.value if hasattr(peer2.verdict, 'value') else str(peer2.verdict)

    consensus_block = ''
    if p1v == p2v:
        consensus_block = (
            f'\n\u26a0\ufe0f  INDEPENDENCE NOTICE: Both other debaters reached the same verdict ({p1v}).'
            f'\nMajority opinion is NOT evidence. Only update your verdict if they cited SPECIFIC'
            f'\nEVIDENCE from the chunks above that you had not considered in Round 0.'
            f'\nAgreeing simply because others agree is intellectual dishonesty.\n'
        )

    return (
        f"USER QUERY: {query}\n\n"
        f"CLAIM TO VERIFY:\n{claim_text[:MAX_CLAIM_CHARS]}\n\n"
        f"RETRIEVED EVIDENCE (same as Round 0 — no new evidence added):\n{format_chunks(chunks)}\n\n"
        f"OTHER DEBATERS' ROUND 0 POSITIONS:\n\n"
        f"[{peer1.debater_label}]\n{fmt_peer(peer1)}\n\n"
        f"[{peer2.debater_label}]\n{fmt_peer(peer2)}\n"
        f"{consensus_block}\n"
        f"YOUR ROUND 1 INSTRUCTIONS:\n"
        f"- Only update your verdict if a debater cited SPECIFIC CHUNK EVIDENCE you had not considered.\n"
        f"- Quote that evidence explicitly if you update.\n"
        f"- If their reasoning is weak or not grounded in the evidence above, push back.\n"
        f"- Maintain your Round 0 position with increased confidence if they failed to cite new evidence.\n\n"
        f"Provide your final verdict."
    )

# Show worst-case context estimate
queries = load_all_queries()
sample_q = max(queries, key=lambda q: len(json.dumps(q['rag_chunks'])))
sample_c = load_claims_for_query(sample_q['query_id'])[0]
r0p = build_round0_prompt(sample_q['user_query'], sample_c['claim_text'], sample_q['rag_chunks'])
print('\u2713 Prompt builders ready')
print(f'  Worst-case R0 prompt: ~{(len(r0p) + len(AGENT_A_SYSTEM))//4} tokens (limit=6144)')

✓ Prompt builders ready
  Worst-case R0 prompt: ~1634 tokens (limit=6144)


In [ ]:
# CELL 12 — Debate Round 0 (3 agents in parallel per claim)

async def debate_claim_round0(session, query: dict, claim: dict) -> dict:
    """Fire all 3 agents simultaneously on one claim.
    Returns: {agent_role: AgentOutputFull}"""
    prompt = build_round0_prompt(
        query['user_query'], claim['claim_text'], query['rag_chunks']
    )

    tasks = {
        role: asyncio.create_task(
            call_agent_with_retry(
                session,
                SYSTEM_PROMPTS[role],
                prompt,
                TEMPERATURES_R0[role],
                role, claim['claim_id'], 0
            )
        )
        for role in AGENT_ROLES
    }

    results = {}
    for role, task in tasks.items():
        parsed, raw, lat, ti, to, retries = await task

        evidence = parsed.get('evidence_cited', [])
        if evidence and isinstance(evidence[0], EvidenceCitation):
            ev_objects = evidence
        else:
            ev_objects = safe_evidence(evidence)

        output = AgentOutputFull(
            agent_role=AgentRole(role),
            round_num=0,
            verdict=AgentVerdict(parsed['verdict']),
            reasoning=parsed.get('reasoning', ''),
            evidence_cited=ev_objects,
            confidence_internal=parsed['confidence_internal'],
        )
        results[role] = output

        write_agent_output(
            claim_id=claim['claim_id'], agent_role=role, round_num=0,
            verdict=output.verdict.value,
            reasoning=output.reasoning,
            evidence_cited=output.evidence_cited,
            confidence_internal=output.confidence_internal,
            raw_response=raw, latency_ms=lat, tokens_in=ti, tokens_out=to,
            parse_failed=parsed.get('_parse_failed', False),
            call_failed=parsed.get('_call_failed', False),
            retry_count=retries,
        )
        run_stats.record(parsed, retries)

        logger.info(
            f'R0 | {claim["claim_id"][:8]} | {role} | '
            f'{output.verdict.value} | conf={output.confidence_internal:.2f} | {lat}ms | retries={retries}'
        )

    return results

print('\u2713 Round 0 function ready')

✓ Round 0 function ready


In [ ]:
# CELL 13 — Debate Round 1 (anti-sycophancy, each agent sees 2 stripped peers)

async def debate_claim_round1(session, query: dict, claim: dict, r0_outputs: dict) -> dict:
    """Round 1: each agent sees the other two agents' R0 outputs (stripped).
    Returns: {agent_role: AgentOutputFull}"""

    stripped = {
        role: strip_for_peer(output, DEBATER_LABEL[role])
        for role, output in r0_outputs.items()
    }

    other_roles = {
        'agent_a': ('agent_b', 'agent_c'),
        'agent_b': ('agent_a', 'agent_c'),
        'agent_c': ('agent_a', 'agent_b'),
    }

    tasks = {}
    for role in AGENT_ROLES:
        p1r, p2r = other_roles[role]
        prompt = build_round1_prompt(
            query['user_query'], claim['claim_text'], query['rag_chunks'],
            peer1=stripped[p1r], peer2=stripped[p2r]
        )
        tasks[role] = asyncio.create_task(
            call_agent_with_retry(
                session, SYSTEM_PROMPTS[role], prompt,
                TEMPERATURES_R1[role], role, claim['claim_id'], 1
            )
        )

    results = {}
    for role, task in tasks.items():
        parsed, raw, lat, ti, to, retries = await task

        evidence = parsed.get('evidence_cited', [])
        ev_objects = evidence if (evidence and isinstance(evidence[0], EvidenceCitation)) else safe_evidence(evidence)

        output = AgentOutputFull(
            agent_role=AgentRole(role),
            round_num=1,
            verdict=AgentVerdict(parsed['verdict']),
            reasoning=parsed.get('reasoning', ''),
            evidence_cited=ev_objects,
            confidence_internal=parsed['confidence_internal'],
        )
        results[role] = output

        # Write R1 output
        write_agent_output(
            claim_id=claim['claim_id'], agent_role=role, round_num=1,
            verdict=output.verdict.value,
            reasoning=output.reasoning,
            evidence_cited=output.evidence_cited,
            confidence_internal=output.confidence_internal,
            raw_response=raw, latency_ms=lat, tokens_in=ti, tokens_out=to,
            parse_failed=parsed.get('_parse_failed', False),
            call_failed=parsed.get('_call_failed', False),
            retry_count=retries,
        )

        # Write delta R0 -> R1
        r0 = r0_outputs[role]
        write_agent_delta(
            claim_id=claim['claim_id'], agent_role=role,
            conf_r0=r0.confidence_internal,
            conf_r1=output.confidence_internal,
            verdict_r0=r0.verdict.value,
            verdict_r1=output.verdict.value,
        )
        run_stats.record(parsed, retries)

        delta = output.confidence_internal - r0.confidence_internal
        flip  = '\u21bb FLIPPED' if r0.verdict.value != output.verdict.value else ''
        logger.info(
            f'R1 | {claim["claim_id"][:8]} | {role} | '
            f'{r0.verdict.value}\u2192{output.verdict.value} {flip} | '
            f'conf {r0.confidence_internal:.2f}\u2192{output.confidence_internal:.2f} (\u0394={delta:+.2f}) | '
            f'{lat}ms | retries={retries}'
        )

    return results

print('\u2713 Round 1 function ready (anti-sycophancy + consensus warning)')

✓ Round 1 function ready (anti-sycophancy + consensus warning)


In [ ]:
# CELL 14 — Per-query orchestrator

async def debate_query(session, query: dict, sem: asyncio.Semaphore,
                       completed_ids: set) -> dict:
    claims = load_claims_for_query(query['query_id'])
    pending = [c for c in claims if c['claim_id'] not in completed_ids]

    if not pending:
        return {'query_id': query['query_id'], 'claims': len(claims),
                'pending': 0, 'errors': 0}

    async def debate_one(claim):
        async with sem:
            try:
                r0 = await debate_claim_round0(session, query, claim)
                r1 = await debate_claim_round1(session, query, claim, r0)
                return 'ok'
            except Exception as e:
                logger.error(f'Claim {claim["claim_id"][:8]} failed: {e}')
                return 'error'

    outcomes = await asyncio.gather(*[debate_one(c) for c in pending])
    errors   = sum(1 for o in outcomes if o == 'error')

    return {'query_id': query['query_id'], 'claims': len(claims),
            'pending': len(pending), 'errors': errors}

print('\u2713 Per-query orchestrator ready')

✓ Per-query orchestrator ready


In [ ]:
# CELL 15 — MAIN RUN (all 50 queries — idempotent, safe to re-run)
# Skips claims that already have complete R1 outputs for all 3 agents.

import asyncio, time, json

async def run_all():
    queries    = load_all_queries()
    sem        = asyncio.Semaphore(CLAIM_CONCURRENCY)
    start_time = time.time()

    # Get already-completed claim ids at start
    completed  = get_completed_claim_ids()
    total_claims = sum(len(load_claims_for_query(q['query_id'])) for q in queries)
    pending_claims = total_claims - len(completed)

    print(f'MAD v2 — {len(queries)} queries | {total_claims} total claims')
    print(f'Already done: {len(completed)} | Pending: {pending_claims}')
    print(f'Expected calls: {pending_claims} x 3 agents x 2 rounds = {pending_claims*6}')
    print(f'Concurrency: {CLAIM_CONCURRENCY} claims x 3 agents = {CLAIM_CONCURRENCY*3} concurrent\n')

    async with aiohttp.ClientSession() as session:
        for i, query in enumerate(queries):
            qid    = query['query_id']
            claims = load_claims_for_query(qid)
            pending_q = [c for c in claims if c['claim_id'] not in completed]

            if not pending_q:
                print(f'[{i+1:02d}/50] {qid} \u2014 all {len(claims)} claims done, skipping')
                continue

            print(f'[{i+1:02d}/50] {qid} | {len(pending_q)}/{len(claims)} claims pending | {query["user_query"][:55]}...')

            try:
                summary = await debate_query(session, query, sem, completed)
                # Update completed set
                completed = get_completed_claim_ids()

                elapsed = time.time() - start_time
                done_q  = i + 1
                rate    = done_q / elapsed if elapsed > 0 else 1
                eta_s   = (len(queries) - done_q) / rate
                print(f'    errors={summary["errors"]} | elapsed={elapsed/60:.1f}min | ETA={eta_s/60:.1f}min')

                # Print live stats every 10 queries
                if done_q % 10 == 0:
                    run_stats.summary()

            except Exception as e:
                logger.error(f'Query {qid} failed: {e}')
            print()

    total = time.time() - start_time
    print(f'\n\u2713 All queries done in {total/60:.1f} minutes')
    run_stats.summary()

asyncio.run(run_all())

MAD v2 — 50 queries | 414 total claims
Already done: 0 | Pending: 414
Expected calls: 414 x 3 agents x 2 rounds = 2484
Concurrency: 4 claims x 3 agents = 12 concurrent

[01/50] q_001 | 11/11 claims pending | What is the relationship between self-reported physical...
    errors=0 | elapsed=0.6min | ETA=29.5min

[02/50] q_002 | 2/2 claims pending | What must be disregarded in the proceeding according to...
    errors=0 | elapsed=0.8min | ETA=19.1min

[03/50] q_003 | 5/5 claims pending | What does 'contrary' mean in the context of comparing S...
    errors=0 | elapsed=1.1min | ETA=17.4min

[04/50] q_004 | 10/10 claims pending | What must a covered entity do when making routine and r...
    errors=0 | elapsed=1.7min | ETA=19.7min

[05/50] q_005 | 9/9 claims pending | What is the requirement for using the Employer Identifi...
    errors=0 | elapsed=2.2min | ETA=19.5min

[06/50] q_006 | 10/10 claims pending | What are the potential effects of Dronedarone on heart ...


    "verdict": "NOT_SUPPORTED",
    "reasoning": "The evid


    errors=0 | elapsed=2.8min | ETA=20.2min

[07/50] q_007 | 6/6 claims pending | What is the recommended dietary protein intake for indi...


    "verdict": "NOT_SUPPORTED",
    "reasoning": "The retr
    "verdict": "NOT_SUPPORTED",
    "reasoning": "The evid
    "verdict": "NOT_SUPPORTED",
    "reasoning": "The evid
    "verdict": "SUPPORTED",
    "reasoning": "The evidence
    "verdict": "NOT_SUPPORTED",
    "reasoning": "The evid


    errors=0 | elapsed=3.2min | ETA=19.7min

[08/50] q_008 | 8/8 claims pending | What are the training requirements for members of a cov...
    errors=0 | elapsed=3.6min | ETA=18.7min

[09/50] q_009 | 10/10 claims pending | What are the requirements for filing a motion for recon...
    errors=0 | elapsed=4.1min | ETA=18.9min

[10/50] q_010 | 13/13 claims pending | What are the criteria for assessing ventricular diastol...
    errors=0 | elapsed=5.0min | ETA=20.0min

=== Run Stats (so far) ===
  Calls: 504  |  Parse fails: 0 (0.0%)  |  Call fails: 0
  Retries used: 6
  Verdicts: {'SUPPORTED': 143, 'PARTIAL': 43, 'NOT_SUPPORTED': 315, 'IDK': 3}
  PARTIAL rate: 8.5% (v1 was 69% — target <50%)

[11/50] q_011 | 12/12 claims pending | What is the recommended protein intake for individuals ...


...

    errors=0 | elapsed=22.2min | ETA=3.0min

[45/50] q_045 | 7/7 claims pending | What are the specific risks associated with GenAI model...


    "verdict": "SUPPORTED",
    "reasoning": "The evidence


    errors=0 | elapsed=22.7min | ETA=2.5min

[46/50] q_046 | 15/15 claims pending | What are the essential elements of a bank's model risk ...
    errors=0 | elapsed=23.6min | ETA=2.0min

[47/50] q_047 | 7/7 claims pending | What must a business provide to a recipient-business wh...


    "verdict": "NOT_SUPPORTED",
    "reasoning": "The evid


    errors=0 | elapsed=23.9min | ETA=1.5min

[48/50] q_048 | 10/10 claims pending | What responsibilities do staff of Member States have in...
    errors=0 | elapsed=24.4min | ETA=1.0min

[49/50] q_049 | 8/8 claims pending | What are the cybersecurity risk considerations for high...
    errors=0 | elapsed=25.0min | ETA=0.5min

[50/50] q_050 | 5/5 claims pending | What are the four principles for explainable AI accordi...
    errors=0 | elapsed=25.2min | ETA=0.0min

=== Run Stats (so far) ===
  Calls: 2484  |  Parse fails: 10 (0.4%)  |  Call fails: 0
  Retries used: 58
  Verdicts: {'SUPPORTED': 605, 'PARTIAL': 336, 'NOT_SUPPORTED': 1528, 'IDK': 15}
  PARTIAL rate: 13.5% (v1 was 69% — target <50%)


✓ All queries done in 25.2 minutes

=== Run Stats (so far) ===
  Calls: 2484  |  Parse fails: 10 (0.4%)  |  Call fails: 0
  Retries used: 58
  Verdicts: {'SUPPORTED': 605, 'PARTIAL': 336, 'NOT_SUPPORTED': 1528, 'IDK': 15}
  PARTIAL rate: 13.5% (v1 was 69% — target <50%)


In [ ]:
# CELL 16 — Verify results
import sqlite3

conn = sqlite3.connect(DB_PATH)

print('=== TABLE COUNTS ===')
for tbl in ['queries', 'claims', 'agent_outputs', 'agent_deltas', 'judge_verdicts']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<20} {n}')

print('\n=== MISSING CLAIMS (if any) ===')
missing = conn.execute("""
    SELECT c.query_id, c.claim_id, c.claim_text
    FROM claims c
    WHERE c.claim_id NOT IN (
        SELECT claim_id FROM agent_outputs WHERE round_num=1
        GROUP BY claim_id HAVING COUNT(DISTINCT agent_role)=3
    )
""").fetchall()
print(f'  Missing: {len(missing)} claims')
for m in missing:
    print(f'  {m[0]} | {m[1][:12]} | {m[2][:60]}')

print('\n=== VERDICT DISTRIBUTION (Round 1) ===')
rows = conn.execute("""
    SELECT agent_role, verdict, COUNT(*) as n
    FROM agent_outputs WHERE round_num=1
    GROUP BY agent_role, verdict ORDER BY agent_role, verdict
""").fetchall()
for r in rows:
    print(f'  {r[0]} | {r[1]:<15} | {r[2]}')

print('\n=== PARTIAL RATE ===')
partial = conn.execute("""
    SELECT ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM agent_outputs WHERE round_num=1),1)
    FROM agent_outputs WHERE round_num=1 AND verdict='PARTIAL'
""").fetchone()[0]
print(f'  Overall PARTIAL rate R1: {partial}%  (v1 was 69% — target <50%)')

print('\n=== CONSENSUS BREAKDOWN (R1) ===')
consensus = conn.execute("""
    WITH per_claim AS (
        SELECT claim_id, COUNT(DISTINCT verdict) as uniq
        FROM agent_outputs WHERE round_num=1 GROUP BY claim_id
    )
    SELECT
        CASE uniq WHEN 1 THEN 'All 3 agree'
                  WHEN 2 THEN '2 agree / 1 differs'
                  WHEN 3 THEN 'All 3 disagree' END as type,
        COUNT(*) as claims,
        ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM per_claim),1) as pct
    FROM per_claim GROUP BY uniq ORDER BY uniq
""").fetchall()
for r in consensus:
    print(f'  {r[0]:<22} {r[1]} claims ({r[2]}%)')

print('\n=== DELTA STATS ===')
deltas = conn.execute("""
    SELECT agent_role,
           ROUND(AVG(ABS(delta)),3) as avg_abs,
           COUNT(CASE WHEN ABS(delta)>0.2 THEN 1 END) as large_shifts,
           COUNT(CASE WHEN verdict_changed=1 THEN 1 END) as flips
    FROM agent_deltas GROUP BY agent_role
""").fetchall()
for r in deltas:
    print(f'  {r[0]} | avg_abs_delta={r[1]} | large_shifts={r[2]} | verdict_flips={r[3]}')

print('\n=== PARSE / CALL FAILURES ===')
fails = conn.execute("""
    SELECT agent_role, round_num,
           SUM(parse_failed) as parse_f, SUM(call_failed) as call_f,
           SUM(retry_count) as retries
    FROM agent_outputs
    GROUP BY agent_role, round_num
    HAVING parse_f>0 OR call_f>0 OR retries>0
""").fetchall()
if fails:
    for r in fails:
        print(f'  {r[0]} R{r[1]} | parse_fails={r[2]} call_fails={r[3]} retries={r[4]}')
else:
    print('  No failures \u2713')

conn.close()

=== TABLE COUNTS ===
  queries              50
  claims               414
  agent_outputs        2484
  agent_deltas         1242
  judge_verdicts       0

=== MISSING CLAIMS (if any) ===
  Missing: 0 claims

=== VERDICT DISTRIBUTION (Round 1) ===
  agent_a | NOT_SUPPORTED   | 271
  agent_a | PARTIAL         | 9
  agent_a | SUPPORTED       | 134
  agent_b | NOT_SUPPORTED   | 331
  agent_b | PARTIAL         | 13
  agent_b | SUPPORTED       | 70
  agent_c | NOT_SUPPORTED   | 268
  agent_c | PARTIAL         | 49
  agent_c | SUPPORTED       | 97

=== PARTIAL RATE ===
  Overall PARTIAL rate R1: 5.7%  (v1 was 69% — target <50%)

=== CONSENSUS BREAKDOWN (R1) ===
  All 3 agree            312 claims (75.4%)
  2 agree / 1 differs    81 claims (19.6%)
  All 3 disagree         21 claims (5.1%)

=== DELTA STATS ===
  agent_a | avg_abs_delta=0.055 | large_shifts=7 | verdict_flips=141
  agent_b | avg_abs_delta=0.107 | large_shifts=51 | verdict_flips=77
  agent_c | avg_abs_delta=0.09 | large_shifts=54

In [ ]:
# CELL 17 — GRPO / SFT / DPO export for Unsloth
# Run this AFTER Stage 2 (judge) to get filled Brier rewards.
# Run now to get the pre-judge GRPO structure (rewards will be None).

import json, sqlite3
from collections import defaultdict

def brier_reward(confidence: float, v_label: float) -> float:
    """Brier-based reward: R = 2*p*v - p^2. Range [-1, 1]."""
    return round(2 * confidence * v_label - confidence ** 2, 4)

def export_for_unsloth(db_path: str, out_prefix: str = '/content/mad_v2'):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    rows = conn.execute("""
        SELECT
            ao.output_id, ao.claim_id, ao.agent_role, ao.round_num,
            ao.verdict, ao.reasoning, ao.confidence_internal, ao.raw_response,
            ao.parse_failed, ao.call_failed, ao.latency_ms,
            c.claim_text, c.is_material, c.is_critical, c.confidence_prior,
            q.user_query, q.rag_chunks, q.baseline_answer, q.query_id,
            jv.v_label, jv.judge_confidence, jv.judge_reasoning
        FROM agent_outputs ao
        JOIN claims  c  USING (claim_id)
        JOIN queries q  USING (query_id)
        LEFT JOIN judge_verdicts jv USING (claim_id)
        ORDER BY q.query_id, c.claim_index, ao.agent_role, ao.round_num
    """).fetchall()
    conn.close()

    grpo_records, sft_records, dpo_pairs = [], [], []
    by_claim = defaultdict(list)
    for r in rows:
        by_claim[r['claim_id']].append(dict(r))

    for claim_id, claim_rows in by_claim.items():
        chunks_raw = json.loads(claim_rows[0]['rag_chunks'])
        chunks_text = '\n\n'.join(
            f"[{c['chunk_id']}] {c['text'][:MAX_CHUNK_CHARS]}" for c in chunks_raw
        )

        for r in claim_rows:
            v_label = r['v_label']
            reward  = brier_reward(r['confidence_internal'], v_label) if v_label is not None else None

            up = (
                f"USER QUERY: {r['user_query']}\n\n"
                f"CLAIM TO VERIFY:\n{r['claim_text']}\n\n"
                f"RETRIEVED EVIDENCE:\n{chunks_text}\n\n"
                f"Provide your verdict."
            )

            grpo_records.append({
                'id':           r['output_id'],
                'query_id':     r['query_id'],
                'claim_id':     claim_id,
                'agent_role':   r['agent_role'],
                'round_num':    r['round_num'],
                'is_material':  bool(r['is_material']),
                'is_critical':  bool(r['is_critical']),
                'system':       SYSTEM_PROMPTS[r['agent_role']],
                'prompt':       up,
                'completion':   r['raw_response'] or '',
                'verdict':      r['verdict'],
                'confidence':   r['confidence_internal'],
                'v_label':      v_label,
                'brier_reward': reward,
                'parse_failed': bool(r['parse_failed']),
                'call_failed':  bool(r['call_failed']),
                'latency_ms':   r['latency_ms'],
            })

            # SFT: high-quality R1 outputs where agent was correct and confident
            if (not r['parse_failed'] and not r['call_failed']
                    and reward is not None and reward > 0.5
                    and r['round_num'] == 1):
                sft_records.append({
                    'system':     SYSTEM_PROMPTS[r['agent_role']],
                    'prompt':     up,
                    'completion': r['raw_response'] or '',
                    'metadata':   {
                        'claim_id':   claim_id,
                        'agent_role': r['agent_role'],
                        'verdict':    r['verdict'],
                        'confidence': r['confidence_internal'],
                        'brier':      reward,
                    }
                })

        # DPO: best vs worst agent on same claim (R1 only, after judge)
        r1s = [r for r in claim_rows
               if r['round_num'] == 1 and r['v_label'] is not None
               and not r['parse_failed'] and not r['call_failed']]

        if len(r1s) >= 2:
            scored = sorted(r1s,
                key=lambda r: brier_reward(r['confidence_internal'], r['v_label']),
                reverse=True)
            winner, loser = scored[0], scored[-1]
            w_reward = brier_reward(winner['confidence_internal'], winner['v_label'])
            l_reward = brier_reward(loser['confidence_internal'],  loser['v_label'])
            if w_reward > l_reward + 0.1:  # meaningful margin
                dpo_pairs.append({
                    'prompt':   (
                        f"USER QUERY: {winner['user_query']}\n\n"
                        f"CLAIM TO VERIFY:\n{winner['claim_text']}\n\n"
                        f"RETRIEVED EVIDENCE:\n{chunks_text}"
                    ),
                    'chosen':   winner['raw_response'] or '',
                    'rejected': loser['raw_response']  or '',
                    'metadata': {
                        'claim_id':      claim_id,
                        'winner_agent':  winner['agent_role'],
                        'loser_agent':   loser['agent_role'],
                        'winner_reward': w_reward,
                        'loser_reward':  l_reward,
                        'v_label':       winner['v_label'],
                    }
                })

    paths = {}
    for name, data in [
        ('grpo', grpo_records),
        ('sft',  sft_records),
        ('dpo',  dpo_pairs),
    ]:
        path = f'{out_prefix}_{name}.jsonl'
        with open(path, 'w') as f:
            for rec in data:
                f.write(json.dumps(rec) + '\n')
        paths[name] = path
        print(f'  {name:<6} {len(data):>5} records  ->  {path}')

    print(f'\n  Note: brier_reward=None until Stage 2 (judge) runs.')
    print(f'  Re-run this cell after judge to fill rewards + build DPO pairs.')
    return paths

print('\u2713 Exporting for Unsloth...')
paths = export_for_unsloth(DB_PATH)

✓ Exporting for Unsloth...
  grpo    2484 records  ->  /content/mad_v2_grpo.jsonl
  sft        0 records  ->  /content/mad_v2_sft.jsonl
  dpo        0 records  ->  /content/mad_v2_dpo.jsonl

  Note: brier_reward=None until Stage 2 (judge) runs.
  Re-run this cell after judge to fill rewards + build DPO pairs.


In [ ]:
# CELL 18 — Download updated DB + export files
from google.colab import files
import os

print('Downloading files...')

# DB
print(f'  DB: {DB_PATH}')
files.download(DB_PATH)

# GRPO export files
for name in ['grpo', 'sft', 'dpo']:
    path = f'/content/mad_v2_{name}.jsonl'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  {name}.jsonl: {size:.1f} KB')
        files.download(path)
    else:
        print(f'  {name}.jsonl not found — run Cell 17 first')

print('\nDone. Next step: Stage 2 (Judge) — run after re-uploading this DB.')

  DB: /content/mad_before_phase1_5090_ragfix_01_.db


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  grpo.jsonl: 15229.6 KB


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  dpo.jsonl: 0.0 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. Next step: Stage 2 (Judge) — run after re-uploading this DB.


## Next Steps After This Notebook

1. **Stage 2 — Judge:** Load `judge_verdicts` table using `Qwen2.5-14B-Instruct-AWQ` as judge (separate cells)
2. **Re-run Cell 17** after judge to fill `brier_reward` and generate DPO pairs
3. **Unsloth GRPO training:** Load `mad_v2_grpo.jsonl`, filter `parse_failed=False`, use `brier_reward` as reward signal

### v2 Fixes Applied
| Fix | Details |
|---|---|
| `awq_marlin` | Faster inference kernel vs plain `awq` |
| `--generation-config vllm` | Disables model's baked-in temp=0.7 override |
| Anti-PARTIAL prompts | Agents must justify PARTIAL; cannot use as hedge |
| Anti-sycophancy R1 | Warns agents when 2 peers agree — pushes independence |
| Retry logic | Up to 3 attempts per call with JSON reminder on retry |
| Confidence clamping | [0.05, 0.95] — preserves Brier reward signal |
| `relevant_quote` optional | Fixes EvidenceCitation crash from v1 |
| Parse tracking | `parse_failed`, `call_failed`, `retry_count` in DB |
| GRPO export | `_grpo.jsonl`, `_sft.jsonl`, `_dpo.jsonl` for Unsloth |